# 采购增量分析

## 1. 读取 processed 明细

In [ ]:
import pandas as pd
import numpy as np
from read_data import read_processed_procurement_inbound

pd.set_option("display.max_columns", None)

结束月 = "2026-09"
df = read_processed_procurement_inbound("2025-10", 结束月)
df

## 2. 时间、范围和金额：df1

排除已确认的历史含税金额缺失记录；限指定板块和外部供应商，近 6 个月含税金额大于 5 万元。


In [ ]:
结束月 = pd.Period(结束月, freq="M")
月份 = pd.period_range(结束月 - 11, 结束月, freq="M")
日期 = pd.to_datetime(df["出入库日期"])
df1 = df.loc[
    日期.between(月份[0].start_time, 结束月.end_time)
    & ~(日期.between("2025-02-07", "2025-07-01") & df["采购金额(万)"].isna())
    & df["板块"].isin(["检验", "检验产品", "诊断"])
    & df["供应商类型"].eq("外部单位")
].copy()
df1["采购月"] = pd.to_datetime(df1["出入库日期"]).dt.to_period("M")
近半年金额 = df1.loc[df1["采购月"].isin(月份[-6:])].groupby("物料编码")["采购金额(万)"].sum()
df1 = df1.loc[df1["物料编码"].isin(近半年金额[近半年金额 > 5].index)].copy()
重点字段 = ["物料编码", "物料名称", "规格", "主单位", "供应商名称", "采购月", "数量", "采购金额(万)", "供应商类型", "板块"]
df1 = df1[重点字段 + [列 for 列 in df1.columns if 列 not in 重点字段]]
print(f"df1：{len(df1):,} 行，{df1['物料编码'].nunique():,} 个物料")
df1

## 3. 月采购量与含税金额

月度按物料和月份汇总；下方展示关联了月度指标的完整过滤明细。补零后的完整月度表在 `月度`。

In [ ]:
月度 = df1.groupby(["物料编码", "采购月"], as_index=False).agg(
    月采购量=("数量", "sum"), 月含税金额万=("采购金额(万)", "sum"))
数量矩阵 = 月度.pivot(index="物料编码", columns="采购月", values="月采购量").reindex(columns=月份).fillna(0)
金额矩阵 = 月度.pivot(index="物料编码", columns="采购月", values="月含税金额万").reindex(columns=月份).fillna(0)
数量矩阵.columns.name = 金额矩阵.columns.name = "采购月"
月度 = pd.concat([数量矩阵.stack().rename("月采购量"), 金额矩阵.stack().rename("月含税金额万")], axis=1).reset_index()
print(f"月度：{len(月度):,} 行")
月度明细 = df1.merge(月度, on=["物料编码", "采购月"], how="left", validate="many_to_one")
重点字段 = ["物料编码", "物料名称", "采购月", "月采购量", "月含税金额万"]
月度明细 = 月度明细[重点字段 + [列 for 列 in 月度明细.columns if 列 not in 重点字段]]
月度明细

## 4. 增长指标

各指标使用同一份月度数据，基期不大于零时留空。

### 最近 3 个月与前 3 个月

In [ ]:
指标 = pd.DataFrame(index=数量矩阵.index)
指标["最近3个月平均采购量"] = 数量矩阵.iloc[:, -3:].mean(axis=1)
指标["前3个月平均采购量"] = 数量矩阵.iloc[:, -6:-3].mean(axis=1)
指标["采购量趋势指数"] = 指标["最近3个月平均采购量"] / 指标["前3个月平均采购量"].where(指标["前3个月平均采购量"] > 0) - 1
明细 = df1.join(指标[["最近3个月平均采购量", "前3个月平均采购量", "采购量趋势指数"]], on="物料编码")
重点字段 = ["物料编码", "物料名称"] + ["最近3个月平均采购量", "前3个月平均采购量", "采购量趋势指数"]
明细 = 明细[重点字段 + [列 for 列 in 明细.columns if 列 not in 重点字段]]
明细

### 最新月环比和连续增长月数

In [ ]:
上期 = 数量矩阵.shift(axis=1)
环比 = 数量矩阵 / 上期.where(上期 > 0) - 1
指标["最近1个月环比增长率"] = 环比.iloc[:, -1]
指标["连续增长月数"] = 环比.gt(0).iloc[:, ::-1].cumprod(axis=1).sum(axis=1).astype(int)
明细 = df1.join(指标[["最近1个月环比增长率", "连续增长月数"]], on="物料编码")
重点字段 = ["物料编码", "物料名称"] + ["最近1个月环比增长率", "连续增长月数"]
明细 = 明细[重点字段 + [列 for 列 in 明细.columns if 列 not in 重点字段]]
明细

### 最近 3 个月占比、半年同比与季度趋势

In [ ]:
全年 = 数量矩阵.sum(axis=1)
指标["最近3个月采购量占比"] = 数量矩阵.iloc[:, -3:].sum(axis=1) / 全年.where(全年 != 0)
前半年 = 数量矩阵.iloc[:, :6].mean(axis=1)
指标["半年同比"] = 数量矩阵.iloc[:, 6:].mean(axis=1) / 前半年.where(前半年 > 0) - 1
季度 = pd.DataFrame({i: 数量矩阵.iloc[:, i*3:i*3+3].mean(axis=1) for i in range(4)})
指标["逐季抬升次数"] = 季度.diff(axis=1).iloc[:, 1:].gt(0).sum(axis=1)
明细 = df1.join(指标[["最近3个月采购量占比", "半年同比", "逐季抬升次数"]], on="物料编码")
重点字段 = ["物料编码", "物料名称"] + ["最近3个月采购量占比", "半年同比", "逐季抬升次数"]
明细 = 明细[重点字段 + [列 for 列 in 明细.columns if 列 not in 重点字段]]
明细

## 5. 核心供应商

从 df1 按 12 个月含税金额计算；金额相同时按供应商名称排序。

In [ ]:
供应商金额 = (df1.groupby(["物料编码", "供应商名称"], as_index=False, dropna=False)["采购金额(万)"].sum()
         .sort_values(["物料编码", "采购金额(万)", "供应商名称"], ascending=[True, False, True]))
供应商 = 供应商金额.drop_duplicates("物料编码").rename(
    columns={"供应商名称": "核心供应商名称", "采购金额(万)": "核心供应商含税金额万"})
累计金额 = df1.groupby("物料编码")["采购金额(万)"].sum().rename("最近12个月含税金额万")
供应商 = 供应商.join(累计金额, on="物料编码")
供应商["单一供应商供货占比"] = 供应商["核心供应商含税金额万"] / 供应商["最近12个月含税金额万"].where(供应商["最近12个月含税金额万"] != 0)
明细 = df1.merge(供应商, on="物料编码", how="left", validate="many_to_one")
重点字段 = ["物料编码", "物料名称", "核心供应商名称", "核心供应商含税金额万", "最近12个月含税金额万", "单一供应商供货占比"]
明细 = 明细[重点字段 + [列 for 列 in 明细.columns if 列 not in 重点字段]]
明细

## 6. 近期采购检查

最近 3 个月每个月的净采购量都必须大于 0；零或负数均不进入最终报表。

In [ ]:
近期非正采购 = 数量矩阵.iloc[:, -3:].le(0).any(axis=1)
近期负净量 = 数量矩阵.iloc[:, -3:].lt(0).any(axis=1)
明细 = df1.loc[df1["物料编码"].isin(近期非正采购[近期非正采购].index)]
print(f"非正采购月：{近期非正采购.sum():,} 个物料；其中负净量：{近期负净量.sum():,} 个物料")
明细

### 负净量物料明细

最近三个月存在净数量为负的月份，不进入候选。

In [ ]:
display(数量矩阵.loc[近期负净量, 月份[-3:]])
明细 = df1.loc[df1["物料编码"].isin(近期负净量[近期负净量].index)]
明细

## 7. 月度汇总报表

一行一个物料，展示窗口内净采购量、含税金额及指标。逐笔字段在 `df1` 查看；多单位时不合计数量。

In [ ]:
属性字段 = ["物料名称", "规格", "主单位"]
def 全部取值(列):
    值 = list(pd.unique(列.dropna()))
    return 值[0] if len(值) == 1 else "多个" + 列.name if 值 else None

属性 = df1.groupby("物料编码", as_index=False, dropna=False)[属性字段].agg(全部取值)
报表 = 属性.merge(指标.reset_index(), on="物料编码", validate="one_to_one").merge(供应商, on="物料编码", validate="one_to_one")
报表 = 报表.loc[~报表["物料编码"].isin(近期非正采购[近期非正采购].index)].copy()
有基线 = 报表["前3个月平均采购量"] > 0
阶跃 = (报表["采购量趋势指数"] >= 0.20) & ((报表["最近1个月环比增长率"] >= 0.10) | (报表["连续增长月数"] >= 2))
爬坡 = (报表["逐季抬升次数"] >= 2) & (报表["半年同比"] >= 0.50)
报表["谈判优先级"] = np.select([有基线 & (阶跃 | 爬坡), 有基线 & (报表["采购量趋势指数"] >= 0.10)], ["优先谈判", "观察"], default="不建议")
总量 = df1.groupby("物料编码")["数量"].sum()
多单位 = 报表["物料编码"].map(df1.groupby("物料编码")["主单位"].nunique().gt(1))
报表["最近12个月采购净量"] = 报表["物料编码"].map(总量).mask(多单位)
报表["业务结束月"] = str(结束月)
顺序 = {"优先谈判": 0, "观察": 1, "不建议": 2}
报表 = 报表.assign(_顺序=报表["谈判优先级"].map(顺序)).sort_values(["_顺序", "采购量趋势指数", "最近12个月含税金额万"], ascending=[True, False, False], na_position="last").drop(columns="_顺序").reset_index(drop=True)
重点字段 = ["业务结束月", "物料编码", "物料名称", "规格", "主单位", "谈判优先级", "核心供应商名称",
    "最近12个月采购净量", "最近12个月含税金额万", "采购量趋势指数", "最近1个月环比增长率", "连续增长月数", "单一供应商供货占比",
    "最近3个月平均采购量", "前3个月平均采购量", "半年同比", "逐季抬升次数", "最近3个月采购量占比",
    "核心供应商含税金额万"]
报表 = 报表[重点字段]
print(f"{len(报表):,} 个物料")
报表

## 8. 月度归档

运行 `uv run python scripts/archive_procurement_increment.py` 默认归档上月；指定月份可重跑历史。